
 Ce notebook implémente un système intelligent capable de :
 - Lire automatiquement des e-mails contenant des tickets
 - Analyser et classifier chaque ticket avec groq (LLM)
 - Mettre à jour un Google Sheet avec les informations extraites
 
 **Objectif** : Traiter 549 tickets et les organiser automatiquement par catégorie et urgence

 Installation des dépendances : 
 
 On commence par installer toutes les bibliothèques nécessaires :
 - `google-auth*` : pour l'authentification OAuth2 avec Google
 - `google-api-python-client` : pour interagir avec Gmail et Google Sheets
 - `requests` : pour appeler l'API groq

In [169]:
# Installation des packages nécessaires
!pip install google-auth google-auth-oauthlib google-auth-httplib2
!pip install google-api-python-client
!pip install groq

  Obtaining dependency information for groq from https://files.pythonhosted.org/packages/2b/64/592078e354946265430f4fbd337271338245531e14504a6c3623dcae59ad/groq-0.34.1-py3-none-any.whl.metadata
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.0/136.0 kB 5.5 MB/s eta 0:00:00


 Imports et configuration initiale
 
 On importe toutes les bibliothèques nécessaires pour :
 - La gestion de l'authentification Google (OAuth2)
 - L'accès aux APIs Gmail et Google Sheets
 - Le décodage des emails (base64)
 - Les appels à l'API groq
 - La sauvegarde des tokens d'authentification

In [23]:

import os
from dotenv import load_dotenv
import base64
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from google.auth.transport.requests import Request
from googleapiclient.discovery import build
import pickle
import requests
import json
from time import sleep
from groq import Groq
import json

# On définit les permissions nécessaires
SCOPES = [
    'https://www.googleapis.com/auth/gmail.readonly',
    'https://www.googleapis.com/auth/spreadsheets'
]

print("Imports réussis")

Imports réussis


 Configuration des identifiants

In [24]:
#Clé API de groq pour pouvoir intéragir avec notre LLM:
load_dotenv()
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
#URL google sheet pour pouvoir le remplir
SPREADSHEET_ID = "1dUf_CJwzO5uF8V0xvw-NmbsY3jN7WKRAZBdDQwi5RWU" 

Définition de la classe EmailTicketAgent :

 Cette classe va encapsulé toute la logique de traitement des e-mails.
 Elle contient les méthodes pour :
 - S'authentifier avec Google
 - Lire les emails
 - Classifier avec groq
 - Mettre à jour le Google Sheet


In [25]:
class EmailTicketAgent:    
    def __init__(self, groq_api_key, spreadsheet_id):
        
        self.groq_api_key = groq_api_key
        self.spreadsheet_id = spreadsheet_id
        self.gmail_service = None
        self.sheets_service = None
        
        # Initialisation client Groq
        self.groq_client = Groq(api_key=self.groq_api_key)
        
        # On crée pour chaque catégorie un onglet dans le google sheet
        self.categories = {
            "Problème technique informatique": "Technique",
            "Demande administrative": "Administrative",
            "Problème d'accès / authentification": "Accès",
            "Demande de support utilisateur": "Support",
            "Bug ou dysfonctionnement d'un service": "Bug"
        }
        
        print("Agent initialisé")

 Méthode d'authentification OAuth2
 
 Cette méthode gère l'authentification avec Google :
 - Vérifie si un token existe déjà (`token.pickle`)
 - Si oui, l'utilise (ou le rafraîchit si expiré)
 - Si non, ouvre une fenêtre de navigateur pour l'authentification
 - Sauvegarde le token pour les prochaines utilisations

In [26]:
def authenticate(self):   
        creds = None
        
        # On vérifie si un token existe déjà
        if os.path.exists('token.pickle'):
            with open('token.pickle', 'rb') as token:
                creds = pickle.load(token)
            print("Token trouvé")
        
        # On valide ou renouvele les credentials
        if not creds or not creds.valid:
            if creds and creds.expired and creds.refresh_token:
                print("Rafraîchissement du token...")
                creds.refresh(Request())
            else:
                print("Ouverture du navigateur pour authentification...")
                flow = InstalledAppFlow.from_client_secrets_file(
                    'credentials.json', SCOPES)
                creds = flow.run_local_server(port=0,prompt='select_account')
            
            # On sauvegarde les credentials
            with open('token.pickle', 'wb') as token:
                pickle.dump(creds, token)
            print("Token sauvegardé")
        
        # On crée les services d'API
        self.gmail_service = build('gmail', 'v1', credentials=creds)
        self.sheets_service = build('sheets', 'v4', credentials=creds)
        print("Authentification réussie - Services Gmail et Sheets prêts")

Méthodes de récupération des emails

Ces méthodes permettent de :
 - Lister tous les emails de la boîte mail
 - Récupérer le contenu complet d'un email spécifique
 - Parser le corps du message

In [27]:
def get_emails(self, max_results=549):
        try:
            # Appel à l'API Gmail pour lister les messages
            results = self.gmail_service.users().messages().list(
                userId='me',  # 'me' = utilisateur authentifié
                maxResults=max_results
            ).execute()
            
            messages = results.get('messages', [])
            print(f"✓ {len(messages)} emails trouvés dans la boîte mail")
            return messages
            
        except Exception as e:
            print(f"✗ Erreur lors de la récupération des emails: {e}")
            return []
    
def get_email_content(self, msg_id):
        try:
            # On récupére le message complet avec tous ses détails
            message = self.gmail_service.users().messages().get(
                userId='me',
                id=msg_id,
                format='full'
            ).execute()
            
            # On extrait le sujet depuis les headers
            headers = message['payload']['headers']
            subject = next(
                (h['value'] for h in headers if h['name'] == 'Subject'),
                'Sans sujet'
            )
            
            # On extrait le corps du message
            body = self._parse_email_body(message['payload'])
            
            return {
                'id': msg_id,
                'subject': subject,
                'body': body
            }
            
        except Exception as e:
            print(f"Erreur lors de la lecture de l'email {msg_id}: {e}")
            return None
    
def _parse_email_body(self, payload):
        body = ""
        
        if 'parts' in payload:
            for part in payload['parts']:
                # Chercher la partie text/plain
                if part['mimeType'] == 'text/plain':
                    if 'data' in part['body']:
                        # Décoder le base64
                        body = base64.urlsafe_b64decode(
                            part['body']['data']
                        ).decode('utf-8')
                        break
                # Récursion pour les parties imbriquées
                elif 'parts' in part:
                    body = self._parse_email_body(part)
                    if body:
                        break
        
        elif 'body' in payload and 'data' in payload['body']:
            body = base64.urlsafe_b64decode(
                payload['body']['data']
            ).decode('utf-8')
        
        return body

Méthode de classification avec groq

Cette méthode utilise l'API groq pour analyser chaque ticket et déterminer :
 - **Catégorie** 
 - **Urgence** 
 - **Synthèse** 
 
 Le prompt est soigneusement conçu pour obtenir une réponse structurée en JSON

In [10]:
context = f"""
Analyse ce ticket de support et fournis UNIQUEMENT un JSON avec cette structure exacte:
{{
    "category": "une parmi: Problème technique informatique, Demande administrative, Problème d'accès / authentification, Demande de support utilisateur, Bug ou dysfonctionnement d'un service",
    "urgency": "une parmi: Anodine, Faible, Modérée, Élevée, Critique",
    "summary": "résumé en 1-2 phrases"
}}
"""

In [28]:
def classify_mail(self, subject, body):
        prompt = f"""
{context}
Sujet: {subject}
Corps: {body[:1000]}
"""
        try:
            response = self.groq_client.chat.completions.create(
                model="llama-3.3-70b-versatile",
                messages=[{"role": "user", "content": prompt}],
                temperature=0.3,
                max_tokens=300
            )

            content = response.choices[0].message.content
            # On nettoie des ``` autour du JSON si présents
            cleaned = content.strip()
            if cleaned.startswith("```json"):
                cleaned = cleaned[7:]
            elif cleaned.startswith("```"):
                cleaned = cleaned[3:]
            if cleaned.endswith("```"):
                cleaned = cleaned[:-3]
            cleaned = cleaned.strip()

            classification = json.loads(cleaned)
            return classification

        except json.JSONDecodeError as e:
            print(f"Erreur parsing JSON: {e}")
            print("Contenu brut:", repr(content))
            print("Contenu nettoyé:", repr(cleaned))
            return None
        except Exception as e:
            print(f"Erreur lors de la classification: {e}")
            return None

 Méthode de mise à jour du Google Sheet

 - Cette méthode ajoute une nouvelle ligne dans la feuille appropriée du Google Sheet.
 - Elle utilise l'API Google Sheets pour append (ajouter à la fin) les données.

In [29]:
def update_sheet(self, category, subject, urgency, summary):
        try:
            # On trouve le nom de la feuille correspondante
            sheet_name = self.categories.get(category)
            if not sheet_name:
                print(f"✗ Catégorie inconnue: {category}")
                return False
            
            # On prépare les données (une ligne avec 3 colonnes)
            values = [[subject, urgency, summary]]
            
            body = {
                'values': values
            }
            
            # On fait appel à l'API Sheets pour ajouter la ligne
            result = self.sheets_service.spreadsheets().values().append(
                spreadsheetId=self.spreadsheet_id,
                range=f'{sheet_name}!A:C',  
                valueInputOption='RAW',  
                insertDataOption='INSERT_ROWS', 
                body=body
            ).execute()
            
            return True
            
        except Exception as e:
            print(f"Erreur lors de la mise à jour du sheet: {e}")
            return False

Méthode principale de traitement
# 
 Cette méthode orchestre tout le workflow :
 1. Récupère tous les emails
 2. Pour chaque email :
    - Lit le contenu
    - Fait analyser par groq
    - Met à jour le Google Sheet
 3. Affiche un rapport final avec les statistiques

In [30]:
def process_all_tickets(self, delay=0.5):
    
        print("démarrage du traitement :")
        print("="*60)
        
        # On récupére tous les emails
        messages = self.get_emails()
        
        if not messages:
            print("Aucun email à traiter")
            return
        
        # On initialise les compteurs
        total = len(messages)
        processed = 0
        errors = 0
        
        # On traite chaque mail de chaque email
        for i, message in enumerate(messages, 1):
            print(f"\n{'─'*60}")
            print(f"[{i}/{total}] Traitement du ticket en cours...")
            
            # Étape 1 : On récupére le contenu de l'email
            email = self.get_email_content(message['id'])
            if not email:
                errors += 1
                continue
            
            # On affiche le sujet 
            print(f"Sujet: {email['subject'][:70]}...")
            
            # Étape 2 : On classifie avec groq
            print(f"Analyse avec groq...")
            classification = self.classify_mail(email['subject'], email['body'])
            
            if not classification:
                errors += 1
                print(f"Échec de la classification")
                continue
            
            # On affiche la classification
            print(f"Catégorie: {classification['category']}")
            print(f"Urgence: {classification['urgency']}")
            print(f"Synthèse: {classification['summary'][:60]}...")
            
            # Étape 3 : On met à jour le Google Sheet
            print(f"Mise à jour du Google Sheet...")
            success = self.update_sheet(
                classification['category'],
                email['subject'],
                classification['urgency'],
                classification['summary']
            )
            
            if success:
                processed += 1
                print(f"Ticket ajouté avec succès")
            else:
                errors += 1
                print(f"Échec de la mise à jour")
            
            # Pause pour éviter les rate limits
            if delay > 0 and i < total:
                sleep(delay)

# On ajoute les méthodes à notre classe
EmailTicketAgent.authenticate = authenticate
EmailTicketAgent.get_emails = get_emails
EmailTicketAgent.get_email_content = get_email_content
EmailTicketAgent._parse_email_body = _parse_email_body
EmailTicketAgent.classify_mail = classify_mail
EmailTicketAgent.update_sheet = update_sheet
EmailTicketAgent.process_all_tickets = process_all_tickets


 Exécution principale
 
 Cette section lance le traitement complet :
 1. Crée une instance de l'agent
 2. S'authentifie avec Google
 3. Lance le traitement de tous les tickets
 
 **Durée estimée** : Environ 5-10 minutes pour 549 emails (selon votre connexion)


In [32]:


# On crée l'instance de l'agent
agent = EmailTicketAgent(GROQ_API_KEY, SPREADSHEET_ID)

# On s'authentifie avec Google
agent.authenticate()

# On lance le traitement de tous les tickets
agent.process_all_tickets(delay=0.5)


def test_sample(agent, n=5):
    """
    Teste le traitement sur seulement N tickets
    Utile pour vérifier que tout fonctionne avant le traitement complet
    
    Args:
        agent: Instance de EmailTicketAgent
        n: Nombre de tickets à traiter
    """
    print(f"🧪 Test sur {n} tickets seulement...\n")
    
    messages = agent.get_emails(max_results=n)
    
    for i, message in enumerate(messages, 1):
        print(f"\n[{i}/{n}] Test du ticket...")
        email = agent.get_email_content(message['id'])
        if email:
            print(f"✓ Email lu: {email['subject'][:50]}... " )
            classification = agent.classify_mail(email['subject'], email['body'])
            if classification:
                print(f"✓ Classification: {classification['category']} - {classification['urgency']}")
            else:
                print("✗ Échec classification")
        else:
            print("✗ Échec lecture email")

# Décommenter pour tester :
####test_sample(agent, n=5)

# %% [markdown]
# ### 📊 Analyse des statistiques par catégorie

# %%
def analyze_sheet_stats(agent):
    """
    Analyse le contenu du Google Sheet et affiche des statistiques
    par catégorie et par urgence
    
    Args:
        agent: Instance de EmailTicketAgent
    """
    print("\n📊 ANALYSE DES STATISTIQUES\n")
    print("="*60)
    
    for category_full, sheet_name in agent.categories.items():
        try:
            # Lire le contenu de chaque feuille
            result = agent.sheets_service.spreadsheets().values().get(
                spreadsheetId=agent.spreadsheet_id,
                range=f'{sheet_name}!A2:C'  # A partir de la ligne 2 (skip header)
            ).execute()
            
            values = result.get('values', [])
            count = len(values)
            
            print(f"\n📂 {sheet_name}: {count} tickets")
            
            # Compter par urgence
            if values:
                urgencies = {}
                for row in values:
                    if len(row) >= 2:
                        urgency = row[1]
                        urgencies[urgency] = urgencies.get(urgency, 0) + 1
                
                for urgency, cnt in sorted(urgencies.items()):
                    print(f"   {urgency}: {cnt}")
        
        except Exception as e:
            print(f"✗ Erreur pour {sheet_name}: {e}")
    
    print("\n" + "="*60)

# Décommenter pour analyser :
analyze_sheet_stats(agent)


Agent initialisé
Token trouvé
Authentification réussie - Services Gmail et Sheets prêts
démarrage du traitement :
✓ 500 emails trouvés dans la boîte mail

────────────────────────────────────────────────────────────
[1/500] Traitement du ticket en cours...
Sujet: Incident de sécurité : fuite de données sensibles détectée...
Analyse avec groq...
Erreur lors de la classification: Error code: 429 - {'error': {'message': "Rate limit reached for project `project_01ka4cmyq8ehbapv1y7fpeapaz` on tokens per minute (TPM): Limit 1000, Used 990, Requested 327. Please try again in 19.02s. You can change your project's rate limits at https://console.groq.com/settings/project/limits", 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
Échec de la classification

────────────────────────────────────────────────────────────
[2/500] Traitement du ticket en cours...
Sujet: Arrêt du serveur de production – impact sur le portail client...
Analyse avec groq...
Catégorie: Problème technique informatique
Urgen